## Imports & Setup

In [1]:
import warnings, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import wandb
from kaggle_secrets import UserSecretsClient

warnings.filterwarnings("ignore")
secrets = UserSecretsClient()
wandb.login(key=secrets.get_secret("WANDB_API_KEY"))

OPTIONS = ["A","B","C","D","E"]
device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
print("Device:", device)
print("Train:", train.shape, "| Test:", test.shape)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: varnitchourasiya27 (varnitchourasiya27-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Device: cuda
Train: (2000, 8) | Test: (500, 7)


## Download & Load GloVe

In [2]:
import os
if not os.path.exists("glove/glove.6B.300d.txt"):
    os.system("wget -q http://nlp.stanford.edu/data/glove.6B.zip")
    os.system("unzip -q glove.6B.zip -d glove")

def load_glove(path="glove/glove.6B.300d.txt"):
    embeddings = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            values = line.split()
            embeddings[values[0]] = np.array(values[1:], dtype=np.float32)
    print(f"Loaded {len(embeddings)} GloVe vectors")
    return embeddings

glove = load_glove()

Loaded 400000 GloVe vectors


## Text → Sequence (word-level)

In [3]:
def text_to_seq(text, glove, max_len=50, dim=300):
    tokens = str(text).lower().split()[:max_len]
    vecs   = [glove.get(t, np.zeros(dim, dtype=np.float32)) for t in tokens]
    while len(vecs) < max_len:
        vecs.append(np.zeros(dim, dtype=np.float32))
    return np.array(vecs, dtype=np.float32)  # (max_len, 300)

## MAP@3 Helper

In [4]:
def map_at_3(df, predict_fn):
    scores = []
    for _, row in df.iterrows():
        preds   = predict_fn(row).split()
        correct = row["answer"]
        score   = 1.0 / (preds.index(correct)+1) if correct in preds else 0.0
        scores.append(score)
    return float(np.mean(scores))

## BiLSTM Model

In [5]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, input_dim=300, hidden_dim=256, num_classes=5):
        super().__init__()
        self.lstm    = nn.LSTM(input_dim, hidden_dim, batch_first=True,
                               bidirectional=True, num_layers=2, dropout=0.3)
        self.dropout = nn.Dropout(0.3)
        self.fc      = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        # x: (batch, seq_len, 300)
        _, (h, _) = self.lstm(x)
        h = torch.cat([h[-2], h[-1]], dim=-1)  # (batch, hidden_dim*2)
        h = self.dropout(h)
        return self.fc(h)  # (batch, 5)

bilstm = BiLSTMClassifier().to(device)
print("Model ready!")
print(f"Parameters: {sum(p.numel() for p in bilstm.parameters()):,}")

Model ready!
Parameters: 2,722,309


## Dataset & DataLoader

In [6]:
class MCQDataset(Dataset):
    def __init__(self, df, glove, max_len=50):
        self.df      = df.reset_index(drop=True)
        self.glove   = glove
        self.max_len = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # concatenate prompt + each option → 5 sequences
        seqs  = []
        for o in OPTIONS:
            combined = str(row["prompt"]) + " " + str(row[o])
            seqs.append(text_to_seq(combined, self.glove, self.max_len))
        seqs  = np.array(seqs, dtype=np.float32)  # (5, max_len, 300)
        label = OPTIONS.index(row["answer"])
        return torch.tensor(seqs), label

class MCQTestDataset(Dataset):
    def __init__(self, df, glove, max_len=50):
        self.df      = df.reset_index(drop=True)
        self.glove   = glove
        self.max_len = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        seqs = []
        for o in OPTIONS:
            combined = str(row["prompt"]) + " " + str(row[o])
            seqs.append(text_to_seq(combined, self.glove, self.max_len))
        return torch.tensor(np.array(seqs, dtype=np.float32))

train_dataset = MCQDataset(train, glove)
loader        = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
print(f"Batches per epoch: {len(loader)}")

Batches per epoch: 63


## Training with W&B Logging

In [7]:
from sklearn.metrics import f1_score as sklearn_f1

optimizer = torch.optim.Adam(bilstm.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
criterion = nn.CrossEntropyLoss()

run = wandb.init(
    entity="varnitchourasiya27-indian-institute-of-technology-madras",
    project="23f3000843-t22026",
    name="bilstm-glove-mcq",
    config={"lr": 1e-3, "epochs": 10, "hidden_dim": 256, "max_len": 50}
)

for epoch in range(10):
    bilstm.train()
    total_loss, all_preds, all_labels = 0, [], []

    for seqs, labels in loader:
        # seqs: (batch, 5, max_len, 300) → process each option separately
        batch_size = seqs.shape[0]
        seqs   = seqs.to(device)      # (batch, 5, max_len, 300)
        labels = labels.to(device)

        # score each option
        logits_list = []
        for i in range(5):
            opt_seq = seqs[:, i, :, :]        # (batch, max_len, 300)
            logit   = bilstm(opt_seq)[:, i]   # score for option i
            logits_list.append(logit)

        scores = torch.stack(logits_list, dim=1)  # (batch, 5)
        loss   = criterion(scores, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bilstm.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        all_preds.extend(scores.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    scheduler.step()
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    f1  = sklearn_f1(all_labels, all_preds, average="macro")
    print(f"Epoch {epoch+1}: loss={total_loss/len(loader):.4f} | acc={acc:.4f} | f1={f1:.4f}")
    wandb.log({"epoch": epoch+1, "loss": total_loss/len(loader), "accuracy": acc, "f1": f1})

wandb.finish()
print("Training done!")

wandb: setting up run zifhu0ka
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260724_080016-zifhu0ka
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bilstm-glove-mcq
wandb: ⭐️ View project at https://wandb.ai/varnitchourasiya27-indian-institute-of-technology-madras/23f3000843-t22026
wandb: 🚀 View run at https://wandb.ai/varnitchourasiya27-indian-institute-of-technology-madras/23f3000843-t22026/runs/zifhu0ka


Epoch 1: loss=1.5050 | acc=0.3300 | f1=0.2964
Epoch 2: loss=1.0182 | acc=0.6005 | f1=0.6007
Epoch 3: loss=0.5926 | acc=0.7880 | f1=0.7873
Epoch 4: loss=0.2136 | acc=0.9365 | f1=0.9349
Epoch 5: loss=0.0987 | acc=0.9685 | f1=0.9685
Epoch 6: loss=0.0527 | acc=0.9860 | f1=0.9860
Epoch 7: loss=0.0161 | acc=0.9995 | f1=0.9995
Epoch 8: loss=0.0091 | acc=1.0000 | f1=1.0000
Epoch 9: loss=0.0057 | acc=1.0000 | f1=1.0000


wandb: updating run metadata


Epoch 10: loss=0.0041 | acc=1.0000 | f1=1.0000


wandb: uploading history steps 8-9, summary, console lines 8-9
wandb: 
wandb: Run history:
wandb: accuracy ▁▄▆▇██████
wandb:    epoch ▁▂▃▃▄▅▆▆▇█
wandb:       f1 ▁▄▆▇██████
wandb:     loss █▆▄▂▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb: accuracy 1
wandb:    epoch 10
wandb:       f1 1
wandb:     loss 0.0041
wandb: 
wandb: 🚀 View run bilstm-glove-mcq at: https://wandb.ai/varnitchourasiya27-indian-institute-of-technology-madras/23f3000843-t22026/runs/zifhu0ka
wandb: ⭐️ View project at: https://wandb.ai/varnitchourasiya27-indian-institute-of-technology-madras/23f3000843-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260724_080016-zifhu0ka/logs


Training done!


## Inference & Val MAP@3

In [8]:
bilstm.eval()

def predict_top3(row):
    seqs = []
    for o in OPTIONS:
        combined = str(row["prompt"]) + " " + str(row[o])
        seqs.append(text_to_seq(combined, glove))
    seqs = torch.tensor(np.array(seqs, dtype=np.float32)).to(device)  # (5, max_len, 300)

    with torch.no_grad():
        scores = []
        for i in range(5):
            s = bilstm(seqs[i].unsqueeze(0))[:, i].item()
            scores.append(s)

    return " ".join([OPTIONS[i] for i in np.argsort(scores)[::-1][:3]])

val_map = map_at_3(train.sample(200, random_state=42), predict_top3)
print(f"BiLSTM Val MAP@3: {val_map:.4f}")

BiLSTM Val MAP@3: 1.0000


## Generate submission.csv

In [9]:
rows = []
for _, row in test.iterrows():
    rows.append({"id": row["id"], "prediction": predict_top3(row)})

sub = pd.DataFrame(rows)
sub.to_csv("submission.csv", index=False)
print(sub.head())
print(f"Saved {len(sub)} rows.")

   id prediction
0   1      A E B
1   2      B A C
2   3      B A C
3   4      E A D
4   5      C D A
Saved 500 rows.
